In [ ]:
# 06c v3 — offline vLLM full-gen + v38b gated selector (Kaggle wait/flag fix)
# Purpose: generate real LoRA raw_output for 600 rows, then run v38b selector.
# IMPORTANT: if vLLM generation fails, this notebook aborts before selector so it cannot create a fake-good summary.
RUN_SERVE=True
BASE_MODEL="/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
TYPE1_LORA="/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1"
TYPE1_PATCH="/kaggle/input/datasets/nguyenminhtric/exact-test/type1_fixed_patch/type1_fixed"
VLLM_HOST="127.0.0.1"; VLLM_PORT=8001
LORA_SERVED_NAME="qwen3-8b-exact-type1"; BASE_SERVED_NAME="qwen3-8b-base"

# Stability-first for Kaggle 2xT4. 2048 is enough for current prompts and avoids long prefill instability.
MAX_MODEL_LEN=2048
GEN_MAX_TOKENS=512  # keep 512 offline to avoid truncating before Final Answer
GEN_TIMEOUT_SEC=90
GEN_REQUEST_RETRIES=2
GEN_ERROR_FAIL_RATE=0.02  # abort if >2% gen errors
MAX_CONSECUTIVE_GEN_ERRORS=3

# Launch attempts. Try CLI attention backend first; if this vLLM build rejects it, fallback attempts will run.
VLLM_LAUNCH_ATTEMPTS=[
    # vLLM 0.22.1 on Kaggle T4 may take >3 min to become ready.
    # Try attention backend + no chunked prefill first. If CLI rejects attention, later attempts fallback.
    {"attention_backend":"TRITON_ATTN", "disable_chunked_prefill":True},
    {"attention_backend":"FLEX_ATTENTION", "disable_chunked_prefill":True},
    {"attention_backend":"XFORMERS", "disable_chunked_prefill":True},
    {"attention_backend":None, "disable_chunked_prefill":True},
    {"attention_backend":"TRITON_ATTN", "disable_chunked_prefill":False},
    {"attention_backend":None, "disable_chunked_prefill":False},
]
print("06c v3 config ready: fail-fast + Kaggle vLLM wait/flag fix enabled")

### 06c v3 Kaggle fix

This version fixes the launcher issue observed in `vllm_06c_v2_attempt5.log`:

- uses `--no-enable-chunked-prefill` instead of invalid `--disable-chunked-prefill`;
- waits up to 7.5 minutes for `/v1/models` because Kaggle 2×T4 TP=2 may take >3 minutes to finish loading/warmup;
- keeps fail-fast generation and selector guards so a fake summary is not produced if vLLM generation fails.


In [ ]:
# Install vLLM (proven ladder, pin 0.22.1) — same as deployment
import os,sys,subprocess
os.environ["TRANSFORMERS_NO_TF"]="1"; os.environ["USE_TF"]="0"; os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"]="python"
def _pip(*a): return subprocess.run([sys.executable,"-m","pip","install","--quiet","--break-system-packages"]+list(a),capture_output=True,text=True)
def _clear():
    for m in list(sys.modules):
        if any(x in m for x in ("vllm","transformers","torch._dynamo","torch._inductor")): del sys.modules[m]
def _imp():
    try:
        from vllm import LLM; return True,""
    except Exception as e: return False,str(e).split("\n")[0][:160]
ok,msg=_imp()
if not ok:
    for tv,vv in [("4.48.0","0.22.1"),("4.44.2","0.22.1"),("4.57.6","0.22.1"),("4.48.0","")]:
        pkg=f"vllm=={vv}" if vv else "vllm"; print("try",tv,pkg,"...",end=" ",flush=True)
        _pip("protobuf==5.29.5"); _pip(f"transformers=={tv}"); _pip(pkg); _clear(); ok,msg=_imp(); print("OK" if ok else "FAIL")
        if ok: break
    if not ok: raise RuntimeError("vLLM install failed")
import vllm; print("vllm",vllm.__version__)

In [ ]:
# vLLM launch + sanity, robust attempts for Kaggle 2xT4 — v3 wait/flag fix
import os, sys, time, subprocess, pathlib, requests, json, shlex
subprocess.run("mkdir -p /usr/local/cuda/lib64 && ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/local/cuda/lib64/libcuda.so 2>/dev/null || true", shell=True)
os.environ.update({
    "CUDA_VISIBLE_DEVICES":"0,1",
    "VLLM_WORKER_MULTIPROC_METHOD":"spawn",
    "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
    "LD_LIBRARY_PATH":"/usr/local/cuda/lib64:/usr/local/nvidia/lib64:"+os.environ.get("LD_LIBRARY_PATH","")
})
try:
    LORA_RANK=int(json.load(open(os.path.join(TYPE1_LORA,"adapter_config.json"))).get("r",16))
except Exception:
    LORA_RANK=16
MAX_LORA_RANK=next(x for x in [8,16,32,64,128] if x>=LORA_RANK)

LOG_DIR=pathlib.Path("/kaggle/working")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def _base_cmd(attention_backend=None, disable_chunked_prefill=True):
    cmd=[sys.executable,"-m","vllm.entrypoints.openai.api_server",
         "--model",BASE_MODEL,
         "--served-model-name",BASE_SERVED_NAME,
         "--dtype","float16",
         "--tensor-parallel-size","2",
         "--max-model-len",str(MAX_MODEL_LEN),
         "--max-num-batched-tokens",str(MAX_MODEL_LEN),
         "--gpu-memory-utilization","0.85",
         "--max-num-seqs","1",
         "--enforce-eager",
         "--disable-custom-all-reduce",
         "--no-enable-prefix-caching",
         "--generation-config","vllm",
         "--enable-lora",
         "--max-lora-rank",str(MAX_LORA_RANK),
         "--lora-modules",f"{LORA_SERVED_NAME}={TYPE1_LORA}",
         "--host",VLLM_HOST,
         "--port",str(VLLM_PORT)]
    if disable_chunked_prefill:
        cmd += ["--no-enable-chunked-prefill"]  # vLLM 0.22.1 flag; not --disable-chunked-prefill
    if attention_backend:
        cmd += ["--attention-backend", attention_backend]
    return cmd

def _tail(path, n=6000):
    try:
        return pathlib.Path(path).read_text(errors="ignore")[-n:]
    except Exception as e:
        return f"<cannot read log: {e}>"

def _models_ok():
    r=requests.get(f"http://{VLLM_HOST}:{VLLM_PORT}/v1/models",timeout=5)
    return r.status_code==200, r.text[:500]

def _completion_ok():
    payload={"model":LORA_SERVED_NAME,"prompt":"Reasoning:\nFinal Answer:","max_tokens":8,"temperature":0.0}
    r=requests.post(f"http://{VLLM_HOST}:{VLLM_PORT}/v1/completions",json=payload,timeout=30)
    return r.status_code==200, r.status_code, r.text[:500]

def launch_attempt(att, attempt_i):
    subprocess.run("pkill -f vllm || true",shell=True)
    time.sleep(5)
    log=str(LOG_DIR/f"vllm_06c_v3_attempt{attempt_i}.log")
    lf=open(log,"w")
    cmd=_base_cmd(attention_backend=att.get("attention_backend"), disable_chunked_prefill=att.get("disable_chunked_prefill",True))
    print("\n=== Launch attempt",attempt_i,"===")
    print(json.dumps(att, ensure_ascii=False))
    print("CMD:", " ".join(shlex.quote(x) for x in cmd))
    p=subprocess.Popen(cmd,stdout=lf,stderr=lf,env=os.environ.copy())
    ready=False
    for i in range(90):  # up to 7.5 minutes; Kaggle TP=2 can take ~3.5+ min to be ready
        time.sleep(5)
        try:
            ok, msg = _models_ok()
            if ok:
                print("/v1/models OK")
                ready=True
                break
        except Exception:
            pass
        if p.poll() is not None:
            print("vLLM exited early. Log tail:\n", _tail(log))
            return None, log
        if i % 6 == 0:
            print("wait",i*5,"sec")
    if not ready:
        print("vLLM not ready within wait window. Log tail:\n", _tail(log))
        try: p.terminate()
        except Exception: pass
        return None, log
    try:
        ok, code, txt = _completion_ok()
        print("/v1/completions sanity:", code, txt[:120])
        if ok:
            return p, log
        print("Completion sanity failed. Log tail:\n", _tail(log))
    except Exception as e:
        print("Completion sanity exception:", repr(e), "\nLog tail:\n", _tail(log))
    try: p.terminate()
    except Exception: pass
    return None, log

VLLM_PROCESS=None; VLLM_LOG=None
if RUN_SERVE:
    for i,att in enumerate(VLLM_LAUNCH_ATTEMPTS,1):
        VLLM_PROCESS,VLLM_LOG=launch_attempt(att,i)
        if VLLM_PROCESS is not None:
            print("vLLM READY with",att,"log",VLLM_LOG)
            break
    if VLLM_PROCESS is None:
        raise RuntimeError("vLLM failed all launch attempts. Inspect /kaggle/working/vllm_06c_v3_attempt*.log")
else:
    ok, txt = _models_ok(); print("existing /v1/models",ok,txt[:200])
    ok, code, txt = _completion_ok(); print("existing /v1/completions",code,txt[:200])
    if not ok:
        raise RuntimeError("Existing vLLM server failed completion sanity")

In [ ]:
# Load + flatten generated_v4style_300 (keep premises_NL for prompt, premises_FOL for v38b)
import json,re,glob,os
def _find():
    for c in ["/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json","/kaggle/input/**/generated_v4style_300.json"]:
        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])
        if h: return h[0]
    return None
def _fol(rec):
    for k in ("premises-FOL","premises_FOL"):
        if k in rec: return rec[k]
    return []
def _nl(rec):
    for k in ("premises-NL","premises_NL","premises"):
        if k in rec: return rec[k]
    return []
def parse_opts(q): return [o[1].strip().replace("\n"," ") for o in re.findall(r"(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)", q, flags=re.S)]
PATH=_find(); RAW=json.load(open(PATH)) if PATH else None
ROWS=[]
for ri,rec in enumerate(RAW or []):
    qs=rec.get("questions") or []; ans=rec.get("answers") or []
    if isinstance(qs,str): qs=[qs]
    for qi,q in enumerate(qs):
        g=str(ans[qi]).strip() if qi<len(ans) else ""
        opts=parse_opts(q); is_mc=(g.upper() in {"A","B","C","D"}) or len(opts)>=2
        ROWS.append({"flat_id":f"generated_v4style_300:{ri}:{qi}","premises_NL":_nl(rec),"premises_FOL":_fol(rec),
                     "question":q,"gold":g,"options":opts,"is_mc":is_mc,"is_ynu":g.strip().title() in {"Yes","No","Unknown","Uncertain"}})
print("dataset",PATH,"| rows",len(ROWS))

In [ ]:
# Build current-row locked prompt (06b style) + full generation via vLLM
# Fail-fast: if vLLM starts returning errors, stop before selector.
import re, time, requests, json, pathlib

def build_prompt_eval(premises, question, options):
    lines=["You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.","","Premises:"]
    for i,p in enumerate(premises,1):
        lines.append(f"{i}. {p}")
    lines += ["","Question:",question]
    is_letter = options and all(re.fullmatch(r"[A-Da-d]",str(o).strip() or "") for o in options)
    hint = "<A, B, C, or D>" if (is_letter or re.search(r"\n\s*[A-D][.)]",question)) else "<Yes, No, or Unknown>"
    lines += ["","Reason step by step briefly, cite supporting premises if useful, and End with exactly one line: Final Answer: "+hint]
    return "\n".join(lines)+"\n"

URL=f"http://{VLLM_HOST}:{VLLM_PORT}/v1/completions"
STOP=["\nReasoning:","\n\nReasoning:","\nQuestion:","\n\nQuestion:","\nPremises:","\n\nPremises:","\n###"]

def gen_one(prompt):
    last_err=None
    for attempt in range(GEN_REQUEST_RETRIES+1):
        t0=time.time()
        try:
            r=requests.post(URL,json={
                "model":LORA_SERVED_NAME,
                "prompt":prompt,
                "max_tokens":GEN_MAX_TOKENS,
                "temperature":0.0,
                "top_p":1.0,
                "stop":STOP,
            },timeout=GEN_TIMEOUT_SEC)
            if r.status_code==200:
                return r.json()["choices"][0]["text"], time.time()-t0
            last_err=f"HTTP {r.status_code}: {r.text[:500]}"
        except Exception as e:
            last_err=repr(e)
        time.sleep(1+attempt)
    raise RuntimeError(last_err)

# Hard sanity before full loop.
print("Generation sanity on first 3 rows...")
for i in range(min(3,len(ROWS))):
    p=build_prompt_eval(ROWS[i].get("premises_NL") or [], ROWS[i]["question"], ROWS[i].get("options") or [])
    raw,sec=gen_one(p)
    print(" sanity",i,"sec",round(sec,2),"text",raw[:120].replace("\n"," "))

# Full generation.
gen_errors=0; consecutive_errors=0
t0=time.time()
for k,r in enumerate(ROWS):
    pr=build_prompt_eval(r.get("premises_NL") or [], r["question"], r.get("options") or [])
    try:
        raw,sec=gen_one(pr)
        consecutive_errors=0
    except Exception as e:
        raw,sec=f"[gen_error] {e}",0.0
        gen_errors += 1
        consecutive_errors += 1
        print(f"GEN_ERROR row={k} consecutive={consecutive_errors}: {e}")
    r["prompt"]=pr; r["raw_output"]=raw; r["latency_sec"]=round(sec,2)

    # Abort early if server is clearly dead.
    if consecutive_errors >= MAX_CONSECUTIVE_GEN_ERRORS:
        pathlib.Path("/kaggle/working/06c_generation_failed_partial.json").write_text(json.dumps(ROWS[:k+1],ensure_ascii=False,indent=1))
        raise RuntimeError(f"vLLM generation failed {consecutive_errors} times consecutively; aborting before selector.")
    if gen_errors > max(3, int(len(ROWS)*GEN_ERROR_FAIL_RATE)):
        pathlib.Path("/kaggle/working/06c_generation_failed_partial.json").write_text(json.dumps(ROWS[:k+1],ensure_ascii=False,indent=1))
        raise RuntimeError(f"Too many generation errors: {gen_errors}/{k+1}; aborting before selector.")

    if (k+1)%50==0:
        elapsed=(time.time()-t0)/60
        print(f"[{k+1}/{len(ROWS)}] elapsed {elapsed:.1f} min | gen_errors={gen_errors}")

health={
    "n":len(ROWS),
    "gen_errors":gen_errors,
    "gen_error_rate":gen_errors/max(len(ROWS),1),
    "elapsed_min":(time.time()-t0)/60,
}
json.dump(health,open("/kaggle/working/06c_generation_health.json","w"),indent=1)
if health["gen_error_rate"] > GEN_ERROR_FAIL_RATE:
    raise RuntimeError(f"Generation error rate too high: {health}")
json.dump(ROWS,open("/kaggle/working/06c_generated_v4style_300_preds.json","w"),ensure_ascii=False,indent=1)
print("saved 06c preds", json.dumps(health,indent=1))

In [ ]:
# v38b engine — v39 canon
import re
# ---------- v39-lite: canonical predicate ----------
def canon_atom(s):
    s=str(s).strip()
    s=re.sub(r'\(x\)|\(\s*x\s*\)','',s)
    s=s.strip()
    # FOL CamelCase atom -> as-is canonical key
    if re.fullmatch(r'[A-Za-z][A-Za-z0-9]*', s):
        return s
    # NL fallback: tokenize, drop stopwords/subjects, light de-inflect, join
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not'}
    toks=re.findall(r"[a-zA-Z]+", s.lower())
    out=[]
    for t in toks:
        if t in STOP: continue
        t=re.sub(r'(ies)$','y',t); t=re.sub(r'(es|s)$','',t); t=re.sub(r'(ing|ed)$','',t)
        out.append(t)
    return "_".join(out) if out else s.lower()

def _norm_tokens(text):
    text=re.sub(r'(?<!^)(?=[A-Z])',' ',str(text))
    toks=re.findall(r'[a-zA-Z]+', text.lower())
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not','one','least','according','premise',
          'premises','following','statement','true','based','above','can','be','inferred','supported','logically'}
    def _stem(t):
        if re.search(r'(ss|us|is)$', t): pass
        elif re.search(r'(ches|shes|xes|zes|ses)$', t): t=t[:-2]
        elif re.search(r'ies$', t): t=t[:-3]+'y'
        elif t.endswith('s'): t=t[:-1]
        t=re.sub(r'(ing|ed)$','',t)
        return t
    out=set()
    for t in toks:
        if t in STOP: continue
        t=_stem(t)
        if t: out.add(t)
    return out

In [ ]:
# FOL parser
# ---------- FOL parser ----------
def parse_fol(fol):
    """Return ('rule',A,B) | ('uni',A) | ('uni_neg',A) | ('exist',A) | ('exist_neg',A) | None"""
    f=str(fol).replace('->','→').replace('¬','~').replace('∀','A').replace('∃','E')
    f=f.strip()
    # implication
    m=re.search(r'\(?\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*→\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*\)?', f)
    if m and f.startswith('A'):
        a=m.group(1).replace(' ',''); b=m.group(2).replace(' ','')
        an=a.startswith('~'); bn=b.startswith('~')
        return ('rule', (canon_atom(a.lstrip('~')),an), (canon_atom(b.lstrip('~')),bn))
    # quantified single atom
    m=re.search(r'^([AE])\s*x?\s*\(?\s*(~?)\s*([A-Za-z0-9]+)\s*\(x\)\s*\)?$', f)
    if m:
        quant,neg,pred=m.group(1),m.group(2)=='~',canon_atom(m.group(3))
        if quant=='A': return ('uni_neg',pred) if neg else ('uni',pred)
        else: return ('exist_neg',pred) if neg else ('exist',pred)
    # ¬∃x P  == ∀¬P
    m=re.search(r'~\s*E\s*x?\s*\(?\s*([A-Za-z0-9]+)\s*\(x\)', f)
    if m: return ('uni_neg',canon_atom(m.group(1)))
    return None

In [ ]:
# closure
# ---------- closure ----------
def build_closure(premises_fol):
    rules=[]; uni=set(); uni_neg=set(); exist=set()
    prov={}  # atom -> premise idx that introduced it (for path)
    for i,fol in enumerate(premises_fol):
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': rules.append((i,p[1],p[2]))
        elif p[0]=='uni': uni.add(p[1]); prov.setdefault(('pos',p[1]),[i])
        elif p[0]=='uni_neg': uni_neg.add(p[1]); prov.setdefault(('neg',p[1]),[i])
        elif p[0]=='exist': exist.add(p[1]); prov.setdefault(('ex',p[1]),[i])
    # forward positive: uni + (A->B, A pos, B pos-polarity) => B uni
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            # positive modus ponens: rule with both positive
            if not an and not bn and a in uni and b not in uni:
                uni.add(b); prov[('pos',b)]=prov.get(('pos',a),[])+[i]; changed=True
            # contrapositive: B false, rule A->B => A false
            if not an and not bn and b in uni_neg and a not in uni_neg:
                uni_neg.add(a); prov[('neg',a)]=prov.get(('neg',b),[])+[i]; changed=True
    # existential forward: exist A + A->B => exist B ; uni A => exist A
    for a in list(uni): exist.add(a); prov.setdefault(('ex',a),prov.get(('pos',a),[]))
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            if not an and not bn and a in exist and b not in exist:
                exist.add(b); prov[('ex',b)]=prov.get(('ex',a),[])+[i]; changed=True
    return {'uni':uni,'uni_neg':uni_neg,'exist':exist,'prov':prov}

In [ ]:
# query+target
# ---------- query type + target ----------
def query_type(q):
    ql=str(q).lower()
    if re.search(r'\bat least one\b|\bsome\b|\bany\b|\bthere (is|exists)\b|does .* one', ql): return 'existential'
    if re.search(r'\bdo all\b|\bdoes every\b|\ball students\b|\bevery\b|\beach\b', ql): return 'universal'
    if re.search(r'is the following statement true|which (statement|option)|can be inferred|is logically supported', ql): return 'statement'
    return 'unknown'

def target_atom(q, atoms):
    qt=_norm_tokens(q)
    scored=[]
    for a in atoms:
        at=_norm_tokens(a)
        if not at: continue
        ov=len(qt & at)/len(at)   # fraction of atom tokens covered by question
        scored.append((ov,len(at & qt),a))
    scored.sort(reverse=True)
    if not scored: return None
    top=scored[0]
    if top[0] < 0.6 or top[1] < 1: return None
    # uniqueness: if a different atom ties on coverage AND raw overlap, ambiguous
    ties=[s for s in scored if abs(s[0]-top[0])<1e-9 and s[1]==top[1] and s[2]!=top[2]]
    if ties: return None
    return top[2]

In [ ]:
# YNU projection + certificate
# ---------- projection with v35 convention + certificate ----------
def prove(premises_fol, question):
    cl=build_closure(premises_fol)
    atoms=cl['uni']|cl['uni_neg']|cl['exist']|{a for _,(a,_),(b,_) in [] }
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    qt=query_type(question); tgt=target_atom(question, allatoms)
    cert={'query_type':qt,'target':tgt,'positive':None,'negative':None,'answer':None,'premises_used':[],'abstain_reason':None}
    if tgt is None:
        cert['answer']=None; cert['abstain_reason']='target_not_matched'; return cert
    pos = tgt in cl['uni'] or tgt in cl['exist']
    neg = tgt in cl['uni_neg']
    cert['positive']=pos; cert['negative']=neg
    if qt=='existential':
        if neg:  # E1: forall-not target -> no instance (convention: wins even under positive conflict)
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='E1_universal_negative'
        elif pos:
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('ex',tgt),cl['prov'].get(('pos',tgt),[])); cert['proof_rule']='PE_witness'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    elif qt=='universal':
        if tgt in cl['uni']:  # PY: positive universal wins
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('pos',tgt),[]); cert['proof_rule']='PY_universal_positive'
        elif neg:
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='U1_universal_negative'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    else:
        cert['answer']=None; cert['abstain_reason']='statement_or_mc_out_of_scope'
    cert['premises_used']=sorted(set(cert['premises_used']))
    return cert

In [ ]:
# MC policy v38b
def classify_option(opt):
    t=str(opt).strip(); tl=t.lower()
    if re.search(r"cannot be (determined|inferred)|undetermined|does not (support|allow)|no conclusion|insufficient", tl):
        return "META"
    # conditional / relative-clause distractor: "X who/that ... must/will/should ..." or "if ... then ..."
    if re.search(r"\bwho\b|\bthat\b", tl) and re.search(r"\b(must|will|should|then)\b", tl): return "CONDITIONAL"
    if re.search(r"^\s*if\b", tl): return "CONDITIONAL"
    if re.search(r"\bmust\b", tl): return "CONDITIONAL"   # malformed "must completes"
    if re.search(r"^\s*no\b", tl): return "UNIV_NEG"
    if re.search(r"^\s*(only some|some only)\b", tl): return "PARTIAL"
    if re.search(r"^\s*(at least one|some|there (is|exists))\b", tl): return "EXIST_POS"
    if re.search(r"^\s*(every|all|each)\b", tl): return "UNIV_POS"
    return "UNKNOWN_OPT"

def allatoms_of(fol):
    A=set()
    for f in fol:
        p=parse_fol(f)
        if not p: continue
        if p[0]=="rule": A.add(p[1][0]); A.add(p[2][0])
        else: A.add(p[1])
    return A

def eval_direct(kind, opt, cl, allatoms):
    atom=target_atom(opt, allatoms)
    if atom is None: return "UNSUPPORTED",None
    if kind=="UNIV_POS": return ("PROVEN" if atom in cl['uni'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="UNIV_NEG": return ("PROVEN" if atom in cl['uni_neg'] else ("DISPROVEN" if atom in cl['uni'] else "UNSUPPORTED")),atom
    if kind=="EXIST_POS": return ("PROVEN" if atom in cl['exist'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="PARTIAL":
        if atom in cl['exist'] and atom not in cl['uni'] and atom not in cl['uni_neg']: return "PROVEN",atom
        return ("DISPROVEN" if (atom in cl['uni'] or atom in cl['uni_neg']) else "UNSUPPORTED"),atom
    return "UNSUPPORTED",atom

def prove_mc_v38b(fol, options):
    cl=build_closure(fol); allatoms=allatoms_of(fol)
    labels=list("ABCD")[:len(options)]
    proven=[]; meta=None; prov_atom=None
    for lab,opt in zip(labels,options):
        k=classify_option(opt)
        if k=="META": meta=lab; continue
        if k in ("CONDITIONAL","UNKNOWN_OPT"): continue   # never selectable
        st,atom=eval_direct(k,opt,cl,allatoms)
        if st=="PROVEN": proven.append((lab,atom))
    cert={'answer':None,'rule':None,'premises_used':[],'abstain_reason':None}
    if len(proven)==1:
        lab,atom=proven[0]; cert['answer']=lab; cert['rule']='MC_unique_direct_proof'
        for key in [('pos',atom),('neg',atom),('ex',atom)]:
            if key in cl['prov']: cert['premises_used']=sorted(set(cl['prov'][key])); break
    elif len(proven)==0 and meta is not None:
        cert['answer']=meta; cert['rule']='MC_meta_cannot_determine'
    else:
        cert['abstain_reason']=('multiple_direct_proven' if proven else 'no_direct_and_no_meta')
    return cert
print('engine ready')

In [ ]:
# Baseline parse + gated selector + eval (baseline vs selected) + save
# Refuse to run if generation failed; prevents fake-good selected_acc from Unknown fallback.
import json, re

def _is_gen_error(raw):
    return str(raw or "").startswith("[gen_error]")

gen_errors=sum(_is_gen_error(r.get("raw_output")) for r in ROWS)
gen_error_rate=gen_errors/max(len(ROWS),1)
print("generation health before selector:", {"gen_errors":gen_errors,"gen_error_rate":gen_error_rate})
if gen_error_rate > GEN_ERROR_FAIL_RATE:
    raise RuntimeError(f"Refusing selector: gen_error_rate={gen_error_rate:.3%} > {GEN_ERROR_FAIL_RATE:.3%}")

def lora_answer(raw):
    if _is_gen_error(raw):
        return "UNPARSEABLE"
    m=re.findall(r"Final Answer:\s*([A-D]|Yes|No|Unknown|Uncertain)\b",raw or "",re.I)
    if not m:
        return "Unknown"
    v=m[0].strip()
    v=v.upper() if re.fullmatch(r"[A-Da-d]",v) else v.title()
    return "Unknown" if v=="Uncertain" else v

def gated_select(row):
    fol=row.get("premises_FOL") or []; q=row.get("question",""); opts=row.get("options") or []
    base=lora_answer(row.get("raw_output",""))
    if fol:
        if row.get("is_mc"):
            c=prove_mc_v38b(fol,opts)
            if c.get("answer") is not None:
                return {"answer":c["answer"],"premises_used":c.get("premises_used",[]),"source":"v38b_mc"}
        else:
            c=prove(fol,q)
            if c.get("answer") is not None:
                return {"answer":c["answer"],"premises_used":c.get("premises_used",[]),"source":"v38b_ynu"}
    return {"answer":base,"premises_used":[],"source":"lora"}

def ng(g):
    g=str(g).strip()
    return g.upper() if g.upper() in {"A","B","C","D"} else g.title()

b=s=n=0; ov=ovg=ovb=pu=0; src={}; out_rows=[]
for r in ROWS:
    gold=ng(r.get("gold")); base=lora_answer(r.get("raw_output","")); o=gated_select(r); a=o["answer"]; n+=1
    b += (base==gold)
    s += (a==gold)
    src[o["source"]]=src.get(o["source"],0)+1
    if o["source"].startswith("v38b"):
        ov += 1
        pu += (1 if o["premises_used"] else 0)
        if a != base:
            if a==gold: ovg += 1
            elif base==gold: ovb += 1
    rr=dict(r); rr.update({"baseline_answer":base,"selected_answer":a,"selected_premises_used":o["premises_used"],"selected_source":o["source"]}); out_rows.append(rr)

summary={
    "n":n,
    "gen_errors":gen_errors,
    "gen_error_rate":round(gen_error_rate,6),
    "baseline_acc":round(b/n,4),
    "selected_acc":round(s/n,4),
    "delta":round((s-b)/n,4),
    "overrides":ov,
    "override_good":ovg,
    "override_bad":ovb,
    "with_certificate":pu,
    "source_dist":src,
    "valid": gen_error_rate <= GEN_ERROR_FAIL_RATE,
}
json.dump(out_rows,open("/kaggle/working/v38b_selected_preds.json","w"),ensure_ascii=False,indent=1)
json.dump(summary,open("/kaggle/working/v38b_selector_summary.json","w"),indent=1)
json.dump({"n":n,"gen_errors":gen_errors,"baseline_acc":summary["baseline_acc"]},open("/kaggle/working/06c_generated_v4style_300_summary.json","w"),indent=1)
print("BASELINE acc",summary["baseline_acc"],"-> SELECTED acc",summary["selected_acc"],"(+%.4f)"%summary["delta"])
print("overrides",ov,"good",ovg,"bad",ovb,"certificate",pu,"| sources",src,"| gen_errors",gen_errors)
print(json.dumps(summary,indent=2))